In [1]:
import pandas as pd
df = pd.read_parquet("../data/processed/issues_clean.parquet")
for i in df.sample(3, random_state=2).index:
    print("RAW  :", (df.loc[i, "title"] + " " + df.loc[i, "body"])[:400].replace("\n", " "))
    print("CLEAN:", df.loc[i, "text_clean"][:400])
    print("-----")
print((df["text_clean"].str.len() < 20).sum(), "very short texts")

RAW  : janitor module conflicts with sklearn cross_val_score ### Describe the bug  Importing janitor when using LeaveOneOut() with cross_val_score() causing warning.  ### Steps/Code to Reproduce  ``` from sklearn.model_selection import LeaveOneOut, KFold, cross_val_score import janitor #cannot be used with cross_val_score since it causes warning from catboost import CatBoostClassifier, Pool cv = LeaveOne
CLEAN: janitor module conflicts with sklearn cross_val_score . Describe the bug Importing janitor when using LeaveOneOut() with cross val score() causing warning. Steps/Code to Reproduce CODEBLOCK Expected Results No warning Actual Results CODEBLOCK Versions CODEBLOCK DeprecationWarning
-----
 Under `Examples`, the second code block does not render correctly. Something to do with the `NORMALIZE_WHITESPACE` comment maybe?learn.metrics.precision_recall_fscore_support.html#sklearn.metrics.precision_recall_fscore_support
CLEAN: precision_recall_fscore_support documentation does not render 

In [2]:
import joblib, pandas as pd
vec, clf = joblib.load("../models/type-tfidf-lr.joblib")
d = df.dropna(subset=["issue_type"])
test = d[d["split"] == "test"].copy()
test["pred"] = clf.predict(vec.transform(test["text_clean"]))

wrong = test[test["pred"] != test["issue_type"]]
print("test errors:", len(wrong), "of", len(test))
print(pd.crosstab(test["issue_type"], test["pred"]))
print(wrong.groupby("repo").size() / test.groupby("repo").size())   # error rate per repo

for _, r in wrong.sample(10, random_state=1).iterrows():
    print(f"[{r['repo']}] true={r['issue_type']} pred={r['pred']} | {r['title'][:100]}")

test errors: 106 of 2482
pred         bug  docs  feature  question
issue_type                               
bug         1606    12       25         2
docs          23   326       11         1
feature       21     3      441         1
question       7     0        0         3
repo
huggingface/transformers     0.009550
microsoft/vscode             0.127119
pandas-dev/pandas            0.037037
scikit-learn/scikit-learn    0.068919
dtype: float64
[scikit-learn/scikit-learn] true=bug pred=docs | duck typed estimators fail in check_estimator
[pandas-dev/pandas] true=feature pred=bug | ENH: Support expressions in `pd.qcut`
[scikit-learn/scikit-learn] true=docs pred=bug | Error on the scikit-learn algorithm cheat-sheet?
[scikit-learn/scikit-learn] true=feature pred=bug | Improve sample_weight handling in sag(a)
[pandas-dev/pandas] true=question pred=bug | BUG: to_datetime does not recast units when passing a DatetimeIndex
[scikit-learn/scikit-learn] true=feature pred=bug | Random forest regr